<a href="https://colab.research.google.com/github/natalianowak1/airbnb-price-optimization/blob/main/airbnb_price_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Przygotowanie środowiska i import bibliotek



In [ ]:
import pandas as pd

# 2. Wczytanie i wstępny przegląd danych

In [ ]:
df = pd.read_excel("/content/Airbnb_Open_Data.xlsx")

Podgląd pierwszych pięciu wierszy

In [ ]:
pd.set_option('display.max_columns', None)
print(df.head())

Identyfikacja zmiennych i weryfikacja typów

In [ ]:
print(df.info())

Zliczanie brakujących wartości (nulli) w każdej kolumnie

In [ ]:
print(df.isnull().sum())

Liczba zduplikowanych wierszy

In [ ]:
print(df.duplicated().sum())

Wnioski ze wstępnej analizy:
1. Metryki ogólne:
*  Tabela zawiera 25 kolumnn o nazwach kolejno: 'id', 'NAME', 'host id', 'host_identity_verified', 'host name', 'neighbourhood group', 'neighbourhood', 'latitude', 'longitude', 'country', 'country code', 'instant_bookable', 'cancellation_policy', 'room type', 'Construction year', 'price', 'service fee', 'minimum nights', 'number of reviews', 'reviews per month', 'review rate number', 'calculated host listings count', 'availability 365', 'house_rules', 'license'
* Tabela liczy 102 599 wierszy (ogłoszeń).
* W zbiorze wykryto 541 całkowicie zduplikowanych wierszy, które należy usunąć.
* Jedyne kolumny, które są w 100% kompletne (0 nulli), to: `id`, `host id` oraz `room type`. W pozostałych 22 kolumnach występują nulle.



2. Zidentyfikowane anomalie i plan ich czyszczenia:
* Zbiór danych charakteryzuje się brakiem spójności w nazwach kolumn (część pisana wielkimi literami, stosowanie spacji na zmianę z podkreśleniami/podłogami). Aby uprosćić kod w Pythonie (zamiast konieczności pisania "df['host id']", można pisać: "df.host_id") oraz zapewnić spójność, nazwy kolumn należy zamienić do formatu `pierwszesłowo_drugiesłowo` (małe litery, spacje zastąpione znakiem `_`).
*   Kolumny takie jak 'price' (cena) oraz 'service fee' (opłata serwisowa) są już zapisane jako dane liczbowe (float64), co umożliwia bezpośrednie wykonywanie obliczeń i analizę statystyczną. Posiadają jednak po 247 braków danych, które trzeba uzupełnić (np. medianą cen).
* Kolumna `instant_bookable` zawiera wartości binarne (1 i 0) oraz 105 nulli. W etapie czyszczenia należy zastąpić nulle wartością `0` (bezpieczne założenie, że brak informacji oznacza brak natychmiastowej rezerwacji), a całą kolumnę zmienić na typ logiczny (`boolean`).
* Kolumna `reviews per month` zawiera aż 15 879 braków danych. Wynika to z faktu, że nowo dodane nieruchomości nie mają jeszcze żadnych opinii. Nulle w tej kolumnie zostaną zastąpione wartością `0.0`.
* Kolumna 'license' zawiera tylko 2 wypełnione wiersze, a cała reszta to nulle (kolumna kwalifikuje się do usunięcia).
* Kolumna `house_rules` zawiera informacje w mniej niż połowie wierszy (52 131 nulli). Braki zostaną zastąpione wartością domyślną "No rules specified".
* Należy wykonać korektę kolumn lokalizacyjnych (`country` i `country code`) Ponieważ cały zbiór danych dotyczy rynku w Nowym Jorku, braki danych w kolumnie kraju i jego kodu zostaną uzupełnione stałymi wartościami (odpowiednio: "United States" oraz "US")
* Kolumna `host_identity_verified` posiada 289 nulli. Zostanie tam wprowadzona kategoria "unconfirmed", co jest bezpieczniejszym podejściem biznesowym niż zakładanie, że profil jest zweryfikowany
* Wykryto po 8 braków danych w pozycjach geograficznych (`latitude`, `longitude`). Ponieważ te współrzędne są kluczowe do poprawnego renderowania map w Tableau, wiersze z tymi brakami zostaną całkowicie usunięte z bazy
* Kolumna `neigbhourhood group` posiada 29 braków. W wierszu gdzie jest brak w `neigbhourhood group` i mamy daną w `neighbourhood`, brak w  `neighbourhood_group` można dobrać na podstawie innych dopasowań, pozostałe wiersze z nullami zostaną zastąpione wartością "Unknown".
* Pozostałe kolumny tekstowe i lokalizacyjne (np. `NAME`, `host name`, `neighbourhood`) posiadają niewielkie liczby braków (od kilku do kilkuset). W ich przypadku wiersze z nullami zostaną zastąpione wartością "Unknown".



# 3. Czyszczenie danych (Data Cleaning)


Usuwanie całkowicie zduplikowanych wierszy

In [ ]:
print(f"Liczba wierszy przed usunięciem duplikatów: {len(df)}")
df = df.drop_duplicates()
print(f"Liczba wierszy po usunięciu duplikatów: {len(df)}\n")

Usuwanie wierszy, które mają nulle w `latitude` lub `longitude`

In [ ]:
print(f"Liczba wierszy przed usunięciem braków w geolokalizacji: {len(df)}")
df = df[(df['latitude'].notnull()) & (df['longitude'].notnull())]
print(f"Liczba wierszy po usunięciu braków w geolokalizacji: {len(df)}")

Usuwanie kolumny `license`

In [ ]:
df = df.drop(columns=['license'])
print(f"Nowa liczba kolumn w tabeli: {len(df.columns)}")

Zmiana nazw kolumn do formatu "snake_case"

In [ ]:
print(list(df.columns))
#zamiana liter z wielkich ma małe
df.columns = df.columns.str.lower()
#zamiana spacji w nazwach kolumn na podkreślenie
df.columns = df.columns.str.replace(' ', '_')
print(list(df.columns))

Zastępowanie braków w kolumnach `name`wartością "Unknown"

In [ ]:
print(f"Liczba nulli w name przed uzupełnieniem: {df.name.isnull().sum()}")
df['name'] = df['name'].fillna("Unknown")
print(f"Liczba nulli w name: {df.name.isnull().sum()}")

Zastępowanie braków w kolumnach `host_name` wartością "Unknown"

In [ ]:
print(f"Liczba nulli w host_name przed uzupełnieniem: {df.host_name.isnull().sum()}\n")
df['host_name'] = df['host_name'].fillna("Unknown")
print(f"Liczba nulli w host_name: {df.host_name.isnull().sum()}")

Sprawdzenie unikalnych wartości dla kolumny `host_identity_verified`: uzupełnienie nulli wartością "unconfirmed".
Założenie, że brak danych oznacza że gospodarz pojawia się w bazie tylko raz i nie ma jak ściągnąć jego statusu po host_id, z resztą przy ilości nulli to jest za dużo pisania kodu względem ilości potencjalnie odzyskanych danych. Założenie, że brak danych oznacza, że gospodarz nie jest zweryfikowany.

In [ ]:
print("Unikalne wartości w host_identity_verified: ", df.host_identity_verified.unique())
print(f"Liczba nulli w host_identity_verified przed uzupełnieniem: {df.host_identity_verified.isnull().sum()}\n")
df['host_identity_verified'] = df['host_identity_verified'].fillna("unconfirmed")
print(f"Liczba nulli w host_identity_verified: {df.host_identity_verified.isnull().sum()}")

Sprawdzenie unikalnych wartości, ujednolicanie wartości i zastępowanie braków w kolumnach `neighbourhood_group` wartością z dopasowań z kolumny `neighbourhood` gdzie jest dana i  wartością "Unknown" tam gdzie w obu kolumnach mamy brak danych.

In [ ]:
print("Unikalne dzielnice główne przed korektą:", df.neighbourhood_group.unique())
#brak ukrytych spacji, ale należy ujednolicić nazwy dzielnic Brooklyn i Manhattan do formatu "Dzielnica" i pozbyć się literówe
neighbourhood_group_repair = {
    'brookln': 'Brooklyn',
    'manhatan': 'Manhattan'
}
df['neighbourhood_group'] = df.neighbourhood_group.replace(neighbourhood_group_repair)
print(f"Unikalne dzielnice główne po korekcie: {df.neighbourhood_group.unique()}\n")

print(f"Liczba nulli w kolumnie neughbourhood_group przed dopasowaniem z neighbourhood: {df.neighbourhood_group.isnull().sum()}")

# Stworzenie słownika mapującego osiedla na dzielnice
neighbourhood_group_dict = df.dropna(subset=['neighbourhood_group', 'neighbourhood']).set_index('neighbourhood')['neighbourhood_group'].to_dict()

# Mapujemy i uzupełniamy brakujące dzielnice główne
df['neighbourhood_group'] = df.neighbourhood_group.fillna(df.neighbourhood.map(neighbourhood_group_dict))
print(f"Liczba nulli w kolumnie neughbourhood_group po dopasowaniu z neighbourhood: {df.neighbourhood_group.isnull().sum()}")
#po tej operacji liczba nulli wynosi 0 więc nie trzeba wykonywać zmiany nulli na "Uknown"

Usuwanie ukrytych spacji i zastępowanie braków w kolumnie `neighbourhood` wartością "Uknown"

In [ ]:
df['neighbourhood'] = df.neighbourhood.str.strip()

print(f"Liczba nulli w kolumnie neighbourhood przed uzupełnieniem: {df.neighbourhood.isnull().sum()}")
df['neighbourhood'] = df['neighbourhood'].fillna("Unknown")
print(f"Liczba nulli w kolumnie neighbourhood po uzupełnieniu: {df.neighbourhood.isnull().sum()}")

Uzupełnianie stałą wartością kolumny `country`

In [ ]:
print("Unikalne kraje: ", df.country.unique())

print(f"Liczba nulli w country przed uzupełnieniem: {df.country.isnull().sum()}")
df['country'] = df['country'].fillna("United States")
print(f"Liczba nulli w country: {df.country.isnull().sum()}")

Uzupełnianie stałą wartością kolumny `country_code`

In [ ]:
print("unikalne kody krajów: ", df.country_code.unique())

print(f"Liczba nulli w country_code przed uzupełnieniem: {df.country_code.isnull().sum()}")
df['country_code'] = df['country_code'].fillna("US")
print(f"Liczba nulli w country_code: {df.country_code.isnull().sum()}")

Uzupełnienie brakujących danych zerami w kolumnie `instant_bookable`

In [ ]:
print(f"Liczba nulli w instant_bookable przed uzupełnieniem zerami: {df['instant_bookable'].isnull().sum()}")
df['instant_bookable'] = df['instant_bookable'].fillna(0)
print(f"Liczba nulli w instant_bookable: {df.instant_bookable.isnull().sum()}")

Zmieniamy typu danych kolumny `instant_bookable` na boolean

In [ ]:
print(f"Typ kolumny przed zmianą: {df.instant_bookable.dtype}")
df['instant_bookable'] = df['instant_bookable'].astype(bool)
print(f"Nowy typ kolumny: {df.instant_bookable.dtype}")

Zastępowanie braków polityki anulowania rezerwacji(`cancellation_policy`) wartością "Unkown"

In [ ]:
print("Unikalne polityki anulowania przed uzupełnieniem: ", df.cancellation_policy.unique())
print(f"Liczba nulli w kolumnie cancellation_policy przed uzupełnieniem: {df.cancellation_policy.isnull().sum()}")
df['cancellation_policy'] = df['cancellation_policy'].fillna("unknown")
print("Unikalne polityki anulowania po uzupełnieniu: ", df.cancellation_policy.unique())
print(f"Liczba nulli w kolumnie cancellation_policy po uzupełnieniu: {df.cancellation_policy.isnull().sum()}")

Zastępowanie braków roku budowy (`construction_year`) medianą oraz zamiana typu danych na liczby całkowite

In [ ]:
print(f"Liczba nulli w kolumnie construction_year przed uzupełnieniem: {df.construction_year.isnull().sum()}")
print(f"pierwotny typ danych w kolumnie construction_year: {df.construction_year.dtype}")
median_construction_year = df.construction_year.median()
df['construction_year'] = df.construction_year.fillna(median_construction_year).astype(int)
print(f"Liczba nulli w kolumnie construction_year po uzupełnieniu: {df.construction_year.isnull().sum()}")
print(f"typ danych w kolumnie construction_year po zmianie: {df.construction_year.dtype}")

Zastępowanie braków w kolumnach `price` i `service_fee` medianą w zależności od `room_type`

In [ ]:
print(f"Liczba nulli w kolumnie price przed uzupełnieniem: {df.price.isnull().sum()}")
print(f"Liczba nulli w kolumnie service_fee przed uzupełnieniem: {df.service_fee.isnull().sum()}\n")


median_price_by_type = df.groupby('room_type')['price'].transform('median')
median_service_fee_by_type = df.groupby('room_type')['service_fee'].transform('median')

df['price'] = df['price'].fillna(median_price_by_type)
df['service_fee'] = df['service_fee'].fillna(median_service_fee_by_type)

print(f"Liczba nulli w kolumnie price: {df.price.isnull().sum()}")
print(f"Liczba nulli w kolumnie service_fee: {df.service_fee.isnull().sum()}")


Zastępowanie braków `minimum_nights` (Minimalna liczba nocy wymagana do rezerwacji) medianą w zależności od `room_type` i zamiana typu danych na liczby całkowite

In [ ]:
print(f"Liczba nulli w kolumnie minimum_nights przed uzupełnieniem: {df.minimum_nights.isnull().sum()}")
print(f"pierwotny typ danych w kolumnie minimum_nights: {df.minimum_nights.dtype}")

median_minimum_nights_by_type = df.groupby('room_type')['minimum_nights'].transform('median')
df['minimum_nights'] = df.minimum_nights.fillna(median_minimum_nights_by_type).astype(int)

print(f"Liczba nulli w kolumnie minimum_nights po uzupełnieniu: {df.minimum_nights.isnull().sum()}")
print(f"typ danych w kolumnie minimum_nights po zmianie: {df.minimum_nights.dtype}\n")

Zastępowanie braków `number_of_reviews` (Łączna liczba recenzji) zerem (założenie że mamy zero recenzji) i zamiana typu danych na liczby całkowite

In [ ]:
print(f"Liczba nulli w kolumnie number_of_reviews przed uzupełnieniem: {df.number_of_reviews.isnull().sum()}")
print(f"pierwotny typ danych w kolumnie number_of_reviews: {df.number_of_reviews.dtype}")

df['number_of_reviews'] = df.number_of_reviews.fillna(0).astype(int)

print(f"Liczba nulli w kolumnie number_of_reviews po uzupełnieniu: {df.number_of_reviews.isnull().sum()}")
print(f"typ danych w kolumnie number_of_reviews po zmianie: {df.number_of_reviews.dtype}\n")

Uzupełnienie nulli w `reviews_per_month` (Średnia liczba recenzji na miesiąc) zerem tam gdzie `number_of_reviews` = 0 oraz wartością wyliczoną z proporcji istniejących danych w pozostałych przypadkach

In [ ]:
print(f"Liczba nulli w reviews per month przed uzupełnieniem: {df.reviews_per_month.isnull().sum()}")

df.loc[df.number_of_reviews == 0, 'reviews_per_month'] = 0.0

# współczynnik recenzji - mediana stosunku łącznej średniej liczby recenzji na miesiąc i łącznej liczby recenzji dla istniejących danych
reviews_ratio = (df.reviews_per_month / df.number_of_reviews).median()

# nadanie wartości nullom iloczynem łącznej liczby recezji i wyliczonego współczynnika
df['reviews_per_month'] = df.reviews_per_month.fillna(df.number_of_reviews * reviews_ratio)

print(f"Liczba nulli w reviews per month: {df.reviews_per_month.isnull().sum()}")

Zastępowanie braków `review_rate_number` (Numeryczna ocena na podstawie recenzji) medianą i zmiana typu danych na liczby całkowite

In [ ]:
print(f"Liczba nulli w kolumnie review_rate_number przed uzupełnieniem: {df.review_rate_number.isnull().sum()}")
print(f"pierwotny typ danych w kolumnie review_rate_number: {df.review_rate_number.dtype}")

median_review_rate_number = df.review_rate_number.median()
df['review_rate_number'] = df.review_rate_number.fillna(median_review_rate_number).astype(int)

print(f"Liczba nulli w kolumnie review_rate_number po uzupełnieniu: {df.review_rate_number.isnull().sum()}")
print(f"typ danych w kolumnie review_rate_number po zmianie: {df.review_rate_number.dtype}\n")

Zamiana braków w `calculated_host_listings_count` (Łączna liczba ogłoszeń gospodarza) policzoną wartością na podstawie `host_id` i zamiana typu danych na liczbny całkowite

In [ ]:
print(f"Liczba nulli w kolumnie calculated_host_listings_count przed uzupełnieniem: {df.calculated_host_listings_count.isnull().sum()}")
print(f"pierwotny typ danych w kolumnie calculated_host_listings_count: {df.calculated_host_listings_count.dtype}")

# Realna liczba ofert dla każdego gospodarza
number_of_calculated_host_listings_count = df.groupby('host_id')['id'].transform('count')

# Uzupełniam nulle tą policzoną serią danych i zamieniamy na int
df['calculated_host_listings_count'] = df.calculated_host_listings_count.fillna(number_of_calculated_host_listings_count).astype(int)

print(f"Liczba nulli w kolumnie calculated_host_listings_count po uzupełnieniu: {df.calculated_host_listings_count.isnull().sum()}")
print(f"typ danych w kolumnie calculated_host_listings_count po zmianie: {df.calculated_host_listings_count.dtype}\n")

Zamiana braków w kolumnie `availability 365` (Liczba dni, w które nieruchomość jest dostępna do rezerwacji w ciągu roku) na miedianę w rozróżnieniu na typ lokalu `room type` i zastąpienie typu danych na liczbę całkowitą

In [ ]:
print(f"Liczba nulli w kolumnie availability_365 przed uzupełnieniem: {df.availability_365.isnull().sum()}")
print(f"pierwotny typ danych w kolumnie availability_365: {df.availability_365.dtype}")

median_availability_365_by_type = df.groupby('room_type')['availability_365'].transform('median')
df['availability_365'] = df.availability_365.fillna(median_availability_365_by_type).astype(int)

print(f"Liczba nulli w kolumnie availability_365 po uzupełnieniu: {df.availability_365.isnull().sum()}")
print(f"typ danych w kolumnie availability_365 po zmianie: {df.availability_365.dtype}\n")

Zastępowanie braków tekstem domyślnym ("No rules specified") w kolumnie `house_rules`

In [ ]:
print(f"Liczba nulli w house_rules przed uzupełnieniem: {df.house_rules.isnull().sum()}\n")
df['house_rules'] = df['house_rules'].fillna("No rules specified")
print(f"Liczba nulli w house_rules: {df.house_rules.isnull().sum()}\n")

In [ ]:
print(df.isnull().sum())